# Colab Training & Evaluation Setup
Run this notebook on Google Colab to prepare the repository, install dependencies, and execute object detection training and evaluation.


In [7]:
# Install required Python packages
!pip install -q torch torchvision torchaudio torchmetrics pycocotools


In [8]:
# Clone the repository if needed and switch to it
import os
repo_dir = '/content/vehicle-damage-triage'
if not os.path.exists(repo_dir):
    !git clone https://github.com/AymanLakhnati/AI-vehicule-damage-triage.git {repo_dir}
else:
    print('Repository already exists at', repo_dir)
os.chdir(repo_dir)
print('Working directory:', os.getcwd())
!ls -la


Repository already exists at /content/vehicle-damage-triage
Working directory: /content/vehicle-damage-triage
total 44
drwxr-xr-x 7 root root  4096 Aug  9 13:43 .
drwxr-xr-x 1 root root  4096 Aug  9 13:44 ..
drwxr-xr-x 8 root root  4096 Aug  9 13:43 .git
-rw-r--r-- 1 root root    61 Aug  9 13:43 .gitignore
drwxr-xr-x 2 root root  4096 Aug  9 13:43 models
drwxr-xr-x 2 root root  4096 Aug  9 13:43 out
-rw-r--r-- 1 root root 11055 Aug  9 13:43 Project_Specs.md
-rw-r--r-- 1 root root     0 Aug  9 13:43 readme.md
drwxr-xr-x 4 root root  4096 Aug  9 13:43 reports
-rw-r--r-- 1 root root     0 Aug  9 13:43 requirments.txt
drwxr-xr-x 3 root root  4096 Aug  9 13:44 src
-rw-r--r-- 1 root root     0 Aug  9 13:43 train_detector_log.txt


# GPU Runtime Setup
Before training, set the Colab runtime to GPU: 1) Runtime → Change runtime type 2) Hardware accelerator → GPU 3) Save.

Then re-run this notebook from the install cell onward.


In [9]:
# Verify dataset paths and mount Google Drive if the dataset is missing
from pathlib import Path
import shutil
import subprocess

repo_root = Path('/content/vehicle-damage-triage')
data_root = repo_root / 'data/raw/cardd/CarDD_release/CarDD_COCO'
print('Data root exists:', data_root.exists())
print('Train annotation exists:', (data_root / 'annotations/instances_train2017.json').exists())
print('Val annotation exists:', (data_root / 'annotations/instances_val2017.json').exists())
print('Train images dir exists:', (data_root / 'train2017').exists())
print('Val images dir exists:', (data_root / 'val2017').exists())

if not data_root.exists() or not (data_root / 'annotations/instances_train2017.json').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('\nMounted Drive. Searching for CarDD dataset in Drive and workspace...')
    proc = subprocess.run(['find', '/content', '-name', 'instances_train2017.json'], capture_output=True, text=True)
    found = proc.stdout.strip().splitlines()
    if found:
        print('Found instances_train2017.json at:')
        for path in found:
            print('  ' + path)
        dataset_file = Path(found[0])
        dataset_dir = dataset_file.parents[1]
        print('\nAssuming dataset root is:', dataset_dir)
        print('Linking or copying dataset into the repo path:', data_root)
        data_root.parent.mkdir(parents=True, exist_ok=True)
        if data_root.exists():
            print('Existing data_root already exists:', data_root)
        else:
            try:
                data_root.symlink_to(dataset_dir, target_is_directory=True)
                print('Created symlink:', data_root, '->', dataset_dir)
            except Exception as exc:
                print('Symlink failed, copying dataset instead:', exc)
                shutil.copytree(dataset_dir, data_root)
                print('Copied dataset to:', data_root)
        print('\nRechecking dataset path existence...')
        print('Data root exists:', data_root.exists())
        print('Train annotation exists:', (data_root / 'annotations/instances_train2017.json').exists())
    else:
        print('No CarDD dataset file found anywhere under /content.')
        print('Please upload the dataset to Colab or mount it from Drive, then rerun this cell.')

    proc2 = subprocess.run(['find', '/content', '-type', 'd', '-name', 'CarDD_COCO'], capture_output=True, text=True)
    dirs = proc2.stdout.strip().splitlines()
    if dirs:
        print('\nFound CarDD_COCO directories:')
        for d in dirs:
            print('  ' + d)
        print('\nIf one of these is the dataset, copy it into the repo path or update the training script paths.')


Mounted at /content/drive

Mounted Drive. Searching for CarDD dataset in Drive and workspace...
No CarDD dataset file found anywhere under /content.
Please upload the dataset to Colab or mount it from Drive, then rerun this cell.


In [ ]:
# Import the CarDD dataset into the expected repo layout if needed
import subprocess
import sys

print('Running import helper...')
subprocess.run([sys.executable, 'src/import_cardd.py'], check=True)
print('Dataset import completed.')


In [10]:
# Colab runtime checks for dataset, checkpoints, and GPU
!find /content -name "instances_train2017.json" 2>/dev/null
!find /content -name "*.pth" 2>/dev/null
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

CUDA: True
GPU: Tesla T4


In [11]:
# Run training
!python src/train_cardd_detector.py


Using device: cuda
Traceback (most recent call last):
  File "/content/vehicle-damage-triage/src/train_cardd_detector.py", line 175, in <module>
    main()
  File "/content/vehicle-damage-triage/src/train_cardd_detector.py", line 98, in main
    train_dataset = CarDDDetectionDataset(TRAIN_ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/vehicle-damage-triage/src/cardd_detection_dataset.py", line 39, in __init__
    raise FileNotFoundError(f"Annotations JSON not found: {self.annotations_json}")
FileNotFoundError: Annotations JSON not found: data/raw/cardd/CarDD_release/CarDD_COCO/annotations/instances_train2017.json


In [12]:
# Run evaluation only if a detector checkpoint exists
from pathlib import Path
checkpoint_dir = Path('models')
checkpoints = sorted(checkpoint_dir.glob('cardd_detector_epoch*.pth'))
print('Detected checkpoint files:')
for ckpt in checkpoints:
    print(' ', ckpt)

if checkpoints:
    import subprocess
    subprocess.run(['python', 'src/evaluate_cardd_detector.py'])
else:
    print('No detector checkpoint found in models/. Train first or copy a checkpoint into this directory.')


Detected checkpoint files:
No detector checkpoint found in models/. Train first or copy a checkpoint into this directory.
